# Final Model used for Kaggle submission 
**Score obtained** : 0.04445

Aris FEVRE - DSBA M2

In [ ]:
import os
import cv2
import numpy as np
import random
import pandas as pd
from pathlib import Path
from sklearn.svm import LinearSVC
from skimage.feature import hog

In [ ]:
#Setting paths
images_dir = os.path.abspath(os.path.join('xray_data', 'images'))
labels_dir = os.path.abspath(os.path.join('xray_data', 'labels'))

print(images_dir)
print(labels_dir)

/Users/mougouman/Desktop/Université/M2/S2/Computer Vision/Kaggle Challenge/xray_data/images
/Users/mougouman/Desktop/Université/M2/S2/Computer Vision/Kaggle Challenge/xray_data/labels


In [ ]:


def yolo_to_xyxy(line, img_w, img_h):
    """
    The goal of this function is to convert yolo format to pixel-based corner coordinates (x1, y1, x2, y2) for bounding boxes.
    We do this because the rest of the pipeline works with image paches and bounding boxes in pixel coordinates, not with normalized coordinates.
    """
    cls, xc, yc, w, h = map(float, line.strip().split())

    #YOLO labels are always normalized to [0,1], so we rescale them to the original image size before converting to corner coordinates.
    xc *= img_w
    yc *= img_h
    w *= img_w
    h *= img_h

    #Now we can convert the center coorfinates (xc,yc) and width/height (w,h) to corner coordinates (x1,y1,x2,y2).
    x1 = int(xc - w / 2)
    y1 = int(yc - h / 2)
    x2 = int(xc + w / 2)
    y2 = int(yc + h / 2)

    #lastly, we make sure that the coordinates are within the image boundaries. This is important to avoid invalid crops and 
    # to ensure that we don't try to access pixels outside the image.
    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(img_w, x2)
    y2 = min(img_h, y2)

    return int(cls), x1, y1, x2, y2

In [ ]:
def extract_hog(img):
    """
    This function extracts HOG features from an image patch. 
    HOG (Histogram of Oriented Gradients) is a feature descriptor that captures the distribution of gradient orientations in localized regions of an image.
    
    The parameters we use are:
    - pixels_per_cell=(8, 8): This means that we compute the histogram of gradients for cells of size 8x8 pixels.
    - cells_per_block=(2, 2): This means that we normalize the histograms over blocks of 2x2 cells (16x16 pixels).
    - feature_vector=True: This means that the output will be a 1D array of features, which is what we want for training our SVM classifier.
    
    Trade-offs: 
    Using smaller cells improves detail but incrzeases feature size and computational time. 
    However, this configuration balances performance and efficiency, especially for a classical ML pipeline like SVM (or SVC in our case).
    """
    return hog(
        img,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        feature_vector=True
    )

In [ ]:
def create_dataset(images_dir, labels_dir, patch_size=64, negatives_per_image=20):
    """
    This function builds a dataset of HOG efatures for classification.
    
    We extract positive samples from ground-truth bounding boxes and add random negative patches to teach the model what non-object regions look like.
    
    This transforms the object detection problem into a patch based classification problem, making it easier for classsical supervised learning mzethods like SVM. 
    """
    X, y = [], []

    image_paths = [p for p in sorted(Path(images_dir).glob("*"))]
    print(f"Found {len(image_paths)} training images in {images_dir}")

    for img_path in image_paths:
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue

        h, w = img.shape
        label_path = Path(labels_dir) / f"{img_path.stem}.txt"
        gt_boxes = []

        if label_path.exists():
            with open(label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue

                    cls, x1, y1, x2, y2 = yolo_to_xyxy(line, w, h)
                    bw = x2 - x1
                    bh = y2 - y1

                    if bw <= 0 or bh <= 0:
                        continue

                    # We want ti add some context around objects to include surrounding cues 
                    # that might help the model capture shapes and textures better.
                    margin_x = int(0.5 * bw)
                    margin_y = int(0.5 * bh)

                    xx1 = max(0, x1 - margin_x)
                    yy1 = max(0, y1 - margin_y)
                    xx2 = min(w, x2 + margin_x)
                    yy2 = min(h, y2 + margin_y)

                    patch = img[yy1:yy2, xx1:xx2]
                    if patch.size == 0:
                        continue

                    #Normalize all patches to a fixed size, for consistent HOG feature extration, 
                    #and to make sure that the SVC receives a fixed-length feature vector.
                    patch = cv2.resize(patch, (patch_size, patch_size))
                    X.append(extract_hog(patch))
                    y.append(cls)
                    gt_boxes.append((x1, y1, x2, y2))

        #Random negatives : the goal is to learn what non-object regions look like. 
        #This is crucial for the SVM to learn how to find a good decision boundary between object and background.
        for _ in range(negatives_per_image):
            if w <= patch_size or h <= patch_size:
                continue

            rx = random.randint(0, w - patch_size)
            ry = random.randint(0, h - patch_size)

            patch = img[ry:ry + patch_size, rx:rx + patch_size]
            if patch.shape != (patch_size, patch_size):
                continue

            X.append(extract_hog(patch))
            y.append(-1)

    return np.array(X), np.array(y)

In [ ]:
def get_candidate_boxes(img):
    #Blur for stabilization and noise reduction.
    blurred = cv2.GaussianBlur(img, (5, 5), 0)

    #Canny edge detection, as learned in class. It is a basic but effective edge detection technique. 
    edges = cv2.Canny(blurred, 50, 150)

    #Find countours thanks to cv2 library. 
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    boxes = []

    for cnt in contours:
        #Establish bounding box for each contour. 
        x, y, w, h = cv2.boundingRect(cnt)
        area = w * h

        #We filter out small contours that are unlikely to be objects, as well as very large ones that are likely to be background or noise.
        if area < 500:  
            continue
        if w < 20 or h < 20:
            continue
        if w > 300 or h > 300:
            continue

        boxes.append((x, y, x+w, y+h))

    return boxes

In [ ]:
#Here we simply create our training dataaset using the create_dataset function we defined above. 
#This will create our feature matrix X_train as well as our label vector y_train. We can then use to train our SVM classifier.
X_train, y_train = create_dataset(
    images_dir="xray_data/images/train",
    labels_dir="xray_data/labels/train",
    patch_size=64,
    negatives_per_image=30
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
#See distribution of classes in the training set. 
print("Classes:", np.unique(y_train, return_counts=True))

Found 4200 training images in xray_data/images/train
X_train shape: (132215, 1764)
y_train shape: (132215,)
Classes: (array([-1,  0,  1,  2,  3,  4,  5]), array([126000,    857,   1115,   1310,   1033,    931,    969]))


In [ ]:
#Model training: we use a LinearSVC, which is a linear Support Vector Machine classifier.
clf = LinearSVC(max_iter=10000, class_weight="balanced")
clf.fit(X_train, y_train)
print("Training done")

Training done


In [ ]:
def detect_with_contours(test_dir, output_dir, clf, patch_size=64):
    """
    This function is used to run inference on test images, by using a contour-based approachn, followed by patch classification with our trained SVM model.

    The pipeline follows this logic:
    1. Generate candidate boxes with contour detection
    2. From each candidate patch, extract HOG features and classify it with the SVM model.
    3. Classify each patch as object or background (-1 for background, class index for object)
    4. Save predictions in YOLO format (class, xc, yc, w, h) in a txt file for each test image.


    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    valid_ext = [".jpg", ".jpeg", ".png", ".bmp"]

    image_paths = [p for p in sorted(Path(test_dir).glob("*")) if p.suffix.lower() in valid_ext]

    print("Nb test images:", len(image_paths))

    for img_path in image_paths:
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue

        h, w = img.shape
        predictions = []

        #Generate candidate regions that may contain an object. 
        #This is a bit more efficient than a sliding window approach, 
        #because it focuses on areas with strong edges and potential object boundaries. 
        boxes = get_candidate_boxes(img)

        for (x1, y1, x2, y2) in boxes:
            patch = img[y1:y2, x1:x2]
            if patch.size == 0:
                continue

            #Resize patches to a fixed size for consistent HOG feature extration. 
            patch = cv2.resize(patch, (patch_size, patch_size))
            feat = extract_hog(patch)

            pred = clf.predict([feat])[0]

            #Class -1 means backgoround, in other words regions without obkects. 
            if pred == -1:
                continue

            #Convert to YOLO format for Kaggle submission.
            xc = ((x1 + x2) / 2) / w
            yc = ((y1 + y2) / 2) / h
            bw = (x2 - x1) / w
            bh = (y2 - y1) / h

            predictions.append([int(pred), xc, yc, bw, bh])

        #Save one prediction .txt file for  each test image. 
        txt_path = output_dir / f"{img_path.stem}.txt"
        with open(txt_path, "w") as f:
            for cls, xc, yc, bw, bh in predictions:
                f.write(f"{cls} {xc} {yc} {bw} {bh}\n")

        print(f"{img_path.stem}: {len(predictions)} preds")

In [ ]:
#Just call the function witht the test images directory. 
detect_with_contours(
    test_dir="xray_data/images/test",
    output_dir="predictions",
    clf=clf
)

Nb test images: 900
000a7a2a11064f729819cc43b467f28f: 4 preds
004d43917a9b4877a52fde68dd779206: 4 preds
00f425b495234635b74a46999e44cc74: 2 preds
01550cdec6024f178fe7d1b36f106f69: 3 preds
0295b9f3d94540d79cae3aebc0cb0eca: 2 preds
029cf286340743299c245917c9fce7ac: 1 preds
034b8ee64b604a12acaad74eda57c208: 7 preds
03765f448e9143068514df5478b2420c: 2 preds
0384efb407924dd5946bf11ca60ed9d2: 0 preds
0393d2edc11b47d6b06f157d89d8d11d: 2 preds
03f39abecc394ec1b623da564b2d51cb: 1 preds
03ffbafcae554ce3be5b206d1fb3e6c9: 3 preds
040ba89e514240e7b00b81474b0fb871: 3 preds
043d38e3db01420289c3fdd879e2aa87: 2 preds
043f41d8c7594943accd9af6bd3c0865: 6 preds
046a1c4469c84082b2ce299ad9a05aa2: 1 preds
0499e318af1b4b15be03bc3fec2847b1: 3 preds
04c8ab60832d4f4d86d03fc30ffb52f7: 1 preds
05050dcd9a054e9ab059dae7fb3aae44: 3 preds
05457c678aab4f3c84aacde9828bee46: 0 preds
05c3dd0e12ce4ff89dfb28e0d262ddd6: 1 preds
05f68c83bccc4d9eb685d0ebacf5f2a9: 5 preds
06875f01279b4279ac5cc33c9e17dc6d: 5 preds
06b9dc4b29054f

In [ ]:
import pandas as pd
from pathlib import Path

def yolo_to_submission_csv(yolo_dir, output_csv):
    """ 
    Function provided by professor to convert YOLO format predictions to a single CSV file for Kaggle submission.
    """
    data = []

    for txt_file in sorted(Path(yolo_dir).glob("*.txt")):
        frame_id = txt_file.stem
        boxes = []

        with open(txt_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls, cx, cy, w, h = map(float, parts)
                    boxes.append([int(cls), cx, cy, w, h])

        data.append({'frame_id': frame_id, 'bbs': str(boxes)})

    pd.DataFrame(data).to_csv(output_csv, index=False)

In [19]:
yolo_to_submission_csv("predictions", "submission.csv")
print("submission.csv created")

submission.csv created
